# 00 - Setup del entorno y descarga de datos

Deja `data/raw/` con la misma muestra de datos en cualquier maquina, para que los notebooks 01-05 sean reproducibles.

## Por que una muestra y no el dataset completo

El dataset completo son 158 GB (68 archivos de `train_landmarks/*.parquet`, ~1.5 GB c/u). `train.csv` ya trae la metadata de las 67,208 secuencias; con 2 archivos de landmarks alcanza para el EDA.

Se descarga con la CLI de `kaggle` y no `kagglehub`: para archivos individuales, `kagglehub` devuelve el contenido comprimido en zip pero con el nombre sin `.zip`, lo que rompe `pd.read_csv`/`read_parquet` en silencio.
- `train.csv`, `supplemental_metadata.csv`, `character_to_prediction_index.json` (pocos MB)
- 2 archivos fijos de `train_landmarks/` (`config.SAMPLE_LANDMARK_PATHS`)

In [1]:
import sys
sys.path.append("..")

from pathlib import Path
from src import config


## 1. Credenciales de Kaggle

Requiere `~/.kaggle/kaggle.json` (API token de https://www.kaggle.com/settings) y haber aceptado las reglas de la competencia: https://www.kaggle.com/competitions/asl-fingerspelling/rules

In [2]:
!pip install -q kaggle



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Descargar metadata completa (train.csv y compania)

Estos archivos vienen comprimidos como `<nombre>.zip` aunque se pida solo uno; `unzip -o` los deja listos y `rm` limpia el zip.

In [3]:
import os
import getpass

# Por si tenemos problemas al ingresar con el token de forma segura. 
# No quedará guardado en el texto del notebook.
os.environ['KAGGLE_API_TOKEN'] = getpass.getpass(prompt='Pega tu token de Kaggle aquí y presiona Enter: ')

In [4]:
config.DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

!kaggle competitions download -c asl-fingerspelling -f train.csv -p "{config.DATA_RAW_DIR}"
!kaggle competitions download -c asl-fingerspelling -f supplemental_metadata.csv -p "{config.DATA_RAW_DIR}"
!kaggle competitions download -c asl-fingerspelling -f character_to_prediction_index.json -p "{config.DATA_RAW_DIR}"


  0%|          | 0.00/1.20M [00:00<?, ?B/s]
 83%|████████▎ | 1.00M/1.20M [00:00<00:00, 3.17MB/s]
100%|██████████| 1.20M/1.20M [00:00<00:00, 3.78MB/s]



  0%|          | 0.00/840k [00:00<?, ?B/s]
100%|██████████| 840k/840k [00:00<00:00, 2.76MB/s]
100%|██████████| 840k/840k [00:00<00:00, 2.74MB/s]



  0%|          | 0.00/405 [00:00<?, ?B/s]
100%|██████████| 405/405 [00:00<00:00, 798kB/s]


In [5]:
import zipfile

# 1. Descomprimir y borrar train.csv.zip
train_zip = config.DATA_RAW_DIR / "train.csv.zip"
if train_zip.exists():
    with zipfile.ZipFile(train_zip, 'r') as zip_ref:
        zip_ref.extractall(config.DATA_RAW_DIR)
    train_zip.unlink() # Esto reemplaza al comando 'rm'

# 2. Descomprimir y borrar supplemental_metadata.csv.zip
supp_zip = config.DATA_RAW_DIR / "supplemental_metadata.csv.zip"
if supp_zip.exists():
    with zipfile.ZipFile(supp_zip, 'r') as zip_ref:
        zip_ref.extractall(config.DATA_RAW_DIR)
    supp_zip.unlink()

## 3. Descargar la muestra fija de landmarks

`config.SAMPLE_LANDMARK_PATHS` es la lista fija de archivos de muestra.

In [6]:
import zipfile
from pathlib import Path

landmarks_dir = config.train_landmarks_dir()
landmarks_dir.mkdir(parents=True, exist_ok=True)

for rel_path in config.SAMPLE_LANDMARK_PATHS:
    fname = Path(rel_path).name
    if (landmarks_dir / fname).exists():
        print("ya existe:", fname)
        continue
    
    # Descarga con comillas en la ruta para evitar el error de los espacios
    !kaggle competitions download -c asl-fingerspelling -f {rel_path} -p "{landmarks_dir}"
    
    # Descomprimir y borrar usando Python en lugar de unzip y rm
    zip_path = landmarks_dir / f"{fname}.zip"
    if zip_path.exists():
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(landmarks_dir)
        zip_path.unlink()

ya existe: 1019715464.parquet
ya existe: 1021040628.parquet


## 4. Verificar el contenido descargado

In [7]:
for p in sorted(config.DATA_RAW_DIR.rglob("*")):
    if p.is_file():
        print(p.relative_to(config.DATA_RAW_DIR), f"({p.stat().st_size / 1e6:.1f} MB)")


.gitkeep (0.0 MB)
character_to_prediction_index.json (0.0 MB)
supplemental_metadata.csv (5.1 MB)
train.csv (5.2 MB)
train_landmarks\1019715464.parquet (1536.0 MB)
train_landmarks\1021040628.parquet (1518.8 MB)


## 5. Probar la carga con `src/data_loading.py`

Si esto corre sin errores, el resto de notebooks (01-05) ya pueden usar `dl.load_train_index()` / `dl.load_landmarks(dl.landmark_path(...))` sin configuracion adicional.

In [8]:
from src import data_loading as dl

train_df = dl.load_train_index()
print(train_df.shape)

sample_landmarks = dl.load_landmarks(dl.landmark_path(config.SAMPLE_LANDMARK_PATHS[0]))
sample_landmarks.shape


(67208, 5)


(161461, 1630)